# Ejercicio 1: de un programa lineal a POO

## ♟️ Plataforma de ajedrez online

Vamos a trabajar con una aplicación de ajedrez online, al estilo de *chess.com* o *lichess*.
En ella:

- Los **usuarios** se registran y tienen una puntuación **ELO** que sube o baja según ganan o pierden.
- Dos usuarios juegan **partidas** entre sí. Cada partida tiene un **ritmo** (*bullet*, *blitz* o *clásica*)
  y el ritmo afecta a cuánto varía el ELO.
- La plataforma mantiene un **ranking**, el **historial** de cada jugador y estadísticas.

A continuación tienes un programa **completamente lineal** (variables sueltas, diccionarios,
listas y funciones) que hace todo esto. **Funciona**, pero tiene todos los problemas que
vimos en el repaso: datos sueltos, lógica duplicada, `if/elif` por cada tipo, nada impide
romper el estado desde fuera...

Tu tarea es **convertirlo paso a paso en un programa orientado a objetos**. En cada paso
crearás una clase nueva que sustituye a una parte del programa lineal. No borres el programa
original: ve construyendo la versión POO en las celdas de cada paso y comprueba al final que
el resultado es el mismo.

---

## El programa lineal

Léelo con calma antes de tocar nada. Ejecútalo y fíjate en la salida: es el comportamiento
que tu versión POO tiene que reproducir.

### Cómo funciona el ELO (versión simplificada)

Para cada partida:

1. Cada jugador obtiene unos **puntos**: `1` si gana, `0.5` si tablas, `0` si pierde.
2. Cada ritmo de partida tiene un **factor K** (cuántos puntos de ELO están en juego):
   las partidas rápidas mueven menos el ELO que las lentas.
3. El **ganador suma K**, el **perdedor resta K** y en **tablas nadie cambia**.

In [2]:
# =========================================================
#  PROGRAMA LINEAL: plataforma de ajedrez online
# =========================================================

# --- "Base de datos" en memoria -------------------------
# Cada jugador es un diccionario suelto dentro de otro diccionario
jugadores = {}

# Cada partida es una tupla: (blancas, negras, resultado, ritmo)
# resultado: "1-0" gana blancas, "0-1" gana negras, "1/2-1/2" tablas
partidas = []

ELO_INICIAL = 1200


# --- Funciones -------------------------------------------
def registrar_jugador(nombre, pais):
    if nombre in jugadores:
        print(f"El jugador {nombre} ya existe")
        return
    jugadores[nombre] = {
        "pais": pais,
        "elo": ELO_INICIAL,
        "victorias": 0,
        "derrotas": 0,
        "tablas": 0,
        "historial": [],
    }


def variacion_elo(puntos, k):
    if puntos == 1:
        return k
    elif puntos == 0:
        return -k
    else:
        return 0


def factor_k(ritmo):
    if ritmo == "bullet":
        return 10
    elif ritmo == "blitz":
        return 20
    elif ritmo == "clasica":
        return 40
    else:
        raise ValueError(f"Ritmo desconocido: {ritmo}")


def tiempo_por_jugador(ritmo):
    if ritmo == "bullet":
        return "1 min"
    elif ritmo == "blitz":
        return "5 min"
    elif ritmo == "clasica":
        return "90 min"
    else:
        raise ValueError(f"Ritmo desconocido: {ritmo}")


def jugar_partida(blancas, negras, resultado, ritmo):
    if blancas not in jugadores or negras not in jugadores:
        raise ValueError("Ambos jugadores deben estar registrados")
    if blancas == negras:
        raise ValueError("Un jugador no puede jugar contra sí mismo")
    if resultado not in ("1-0", "0-1", "1/2-1/2"):
        raise ValueError(f"Resultado inválido: {resultado}")

    k = factor_k(ritmo)

    if resultado == "1-0":
        puntos_b, puntos_n = 1, 0
        jugadores[blancas]["victorias"] += 1
        jugadores[negras]["derrotas"] += 1
    elif resultado == "0-1":
        puntos_b, puntos_n = 0, 1
        jugadores[blancas]["derrotas"] += 1
        jugadores[negras]["victorias"] += 1
    else:
        puntos_b, puntos_n = 0.5, 0.5
        jugadores[blancas]["tablas"] += 1
        jugadores[negras]["tablas"] += 1

    jugadores[blancas]["elo"] += variacion_elo(puntos_b, k)
    jugadores[negras]["elo"] += variacion_elo(puntos_n, k)

    partida = (blancas, negras, resultado, ritmo)
    partidas.append(partida)
    jugadores[blancas]["historial"].append(partida)
    jugadores[negras]["historial"].append(partida)


def partidas_jugadas(nombre):
    j = jugadores[nombre]
    return j["victorias"] + j["derrotas"] + j["tablas"]


def porcentaje_victorias(nombre):
    total = partidas_jugadas(nombre)
    if total == 0:
        return 0.0
    return 100 * jugadores[nombre]["victorias"] / total


def mostrar_ranking():
    print("\nRANKING")
    ordenados = sorted(jugadores.items(), key=lambda par: par[1]["elo"], reverse=True)
    for pos, (nombre, datos) in enumerate(ordenados, start=1):
        print(
            f"{pos}. {nombre:<8} ({datos['pais']})  ELO {datos['elo']}  "
            f"| {datos['victorias']}V {datos['derrotas']}D {datos['tablas']}T "
            f"| {porcentaje_victorias(nombre):.0f}% victorias"
        )


def mostrar_historial(nombre):
    print(f"\nHistorial de {nombre}")
    for blancas, negras, resultado, ritmo in jugadores[nombre]["historial"]:
        rival = negras if blancas == nombre else blancas
        color = "blancas" if blancas == nombre else "negras"
        print(f"  vs {rival:<8} con {color:<8} {resultado:<8} [{ritmo}, {tiempo_por_jugador(ritmo)}]")


# --- Programa principal -----------------------------------
registrar_jugador("Magnus", "NOR")
registrar_jugador("Hikaru", "USA")
registrar_jugador("Ding", "CHN")
registrar_jugador("Judit", "HUN")
registrar_jugador("Magnus", "NOR")  # duplicado

jugar_partida("Magnus", "Hikaru", "1-0", "blitz")
jugar_partida("Ding", "Judit", "1/2-1/2", "clasica")
jugar_partida("Hikaru", "Ding", "1-0", "bullet")
jugar_partida("Judit", "Magnus", "0-1", "clasica")
jugar_partida("Hikaru", "Judit", "0-1", "blitz")
jugar_partida("Magnus", "Ding", "1/2-1/2", "bullet")

mostrar_ranking()
mostrar_historial("Magnus")
mostrar_historial("Judit")

print(f"\nTotal de partidas en la plataforma: {len(partidas)}")

El jugador Magnus ya existe

RANKING
1. Magnus   (NOR)  ELO 1260  | 2V 0D 1T | 67% victorias
2. Ding     (CHN)  ELO 1190  | 0V 1D 2T | 0% victorias
3. Judit    (HUN)  ELO 1180  | 1V 1D 1T | 33% victorias
4. Hikaru   (USA)  ELO 1170  | 1V 2D 0T | 33% victorias

Historial de Magnus
  vs Hikaru   con blancas  1-0      [blitz, 5 min]
  vs Judit    con negras   0-1      [clasica, 90 min]
  vs Ding     con blancas  1/2-1/2  [bullet, 1 min]

Historial de Judit
  vs Ding     con negras   1/2-1/2  [clasica, 90 min]
  vs Magnus   con blancas  0-1      [clasica, 90 min]
  vs Hikaru   con negras   0-1      [blitz, 5 min]

Total de partidas en la plataforma: 6


### ¿Qué problemas ves?

Antes de empezar con el ejercicio, vamos a pensar qué problemas de diseño tiene este programa. Piensa en: dónde viven los datos, quién puede modificarlos, qué pasa si
añadimos un nuevo ritmo de partida (*rapid*, por ejemplo), qué pasa si nos equivocamos al
escribir una clave del diccionario...

---

## Paso 1: la clase `Jugador`

**Concepto: clase, `__init__`, `self`, atributos y métodos de instancia.**

Sustituye el diccionario de cada jugador por una clase `Jugador`:

- Atributos de instancia: `nombre`, `pais`, `elo` (empieza en 1200), `victorias` (int), `derrotas` (int),
  `tablas` (int) y `historial` (lista vacía).
- Métodos de instancia:
  - `partidas_jugadas()` → número total de partidas.
  - `porcentaje_victorias()` → porcentaje de victorias (0.0 si no ha jugado).
  - `registrar_resultado(puntos)` → recibe `1`, `0.5` o `0` y suma 1 al contador que toque.
  - `resumen()` → devuelve el string `"Magnus (NOR)  ELO 1200  | 2V 0D 1T | 67% victorias"`
    (es el trozo de línea que imprime `mostrar_ranking` para cada jugador).

Fíjate en que `partidas_jugadas(nombre)` y `porcentaje_victorias(nombre)` recibían el nombre
y buscaban en el diccionario global. Ahora **el propio objeto ya tiene sus datos**: no hace
falta pasarle nada.

> ⚠️ `historial` es una lista (mutable). Recuerda dónde tiene que inicializarse.

In [1]:
class Jugador:
    def __init__(self, nombre, pais):
        self.nombre = nombre
        self.pais = pais
        self.elo = 1200
        self.victorias = 0
        self.derrotas = 0
        self.tablas = 0
        self.historial = []

    def partidas_jugadas(self):
        return self.victorias + self.derrotas + self.tablas

    def porcentaje_victorias(self):
        total = self.partidas_jugadas()
        if total:
            return 100 * self.victorias / total
        return 0

    def registrar_resultado(self, puntos):
        if puntos == 1:
            self.victorias += 1
        elif puntos == 0:
            self.derrotas += 1
        else:
            self.tablas += 1

    def resumen(self):
        mensaje = f"{self.nombre} ({self.pais})  ELO {self.elo} | {self.victorias}V {self.derrotas}D {self.tablas}T | {self.porcentaje_victorias():.0f}% victorias"
        return mensaje

# Prueba
m = Jugador("Magnus", "NOR")
m.registrar_resultado(1)
m.registrar_resultado(1)
m.registrar_resultado(0.5)
print(m.resumen())
# Esperado: Magnus (NOR) ELO 1200 | 2V 0D 1T | 67% victorias
# Esperado: Magnus   (NOR)  ELO 1200  | 2V 0D 1T | 67% victorias

Magnus (NOR)  ELO 1200 | 2V 0D 1T | 67% victorias


---

## Paso 2: varios objetos, cada uno con su estado

**Concepto: cada instancia tiene su propio estado.**

Crea cuatro jugadores (`Magnus`, `Hikaru`, `Ding` y `Judit`) y guárdalos en una **lista**.
Registra a mano un par de resultados distintos en cada uno y comprueba, recorriendo la lista,
que **los contadores de uno no afectan a los demás**.

Después, haz lo siguiente **sin tocar la clase**:

1. Ordena la lista por ELO descendente usando `sorted()` con `key=`.
2. Busca el jugador con más victorias con `max()` y `key=`.

In [ ]:
jugador_1 = Jugador("Magnus", "España")
jugador_2 = Jugador("Hikaru", "Japón")
jugador_3 = Jugador("Ding", "Brasil")
jugador_4 = Jugador("Judit", "Fracnia")

jugador_1.registrar_resultado(1)
jugador_2.registrar_resultado(0)


lista_jugadores = [jugador_1, jugador_2, jugador_3, jugador_4]

#1.
lista_ordenada = sorted(lista_jugadores, key=lambda x: x.elo)
jugador_mas_victorias = max(lista_jugadores, key=lambda x: x.victorias)

#2.

for jugador in lista_jugadores:
    print(jugador.resumen())
print("-------------------------")    
print(jugador_mas_victorias)
print("-------------------------")
print(jugador_mas_victorias.nombre)





Magnus (España)  ELO 1200 | 1V 0D 0T | 100% victorias
Hikaru (Japón)  ELO 1200 | 0V 1D 0T | 0% victorias
Ding (Brasil)  ELO 1200 | 0V 0D 0T | 0% victorias
Judit (Fracnia)  ELO 1200 | 0V 0D 0T | 0% victorias
-------------------------
-------------------------
Magnus


---

## Paso 3: atributos de clase

**Concepto: atributos de clase vs atributos de instancia.**

La constante `ELO_INICIAL` es un dato que pertenece a *todos* los jugadores, no a uno en
concreto. Muévela dentro de la clase como **atributo de clase** y úsala en `__init__`.

Además, añade un atributo de clase `total_registrados` que cuente cuántos jugadores se han
creado. Cada vez que se ejecute `__init__`, debe incrementarse.

In [14]:
class Jugador:
    
    ELO_INICIAL = 1200
    total_registrados = 0
    
    def __init__(self, nombre, pais):
        self.nombre = nombre
        self.pais = pais
        self.elo = Jugador.ELO_INICIAL
        self.victorias = 0
        self.derrotas = 0
        self.tablas = 0
        self.historial = []
        Jugador.total_registrados += 1

    def partidas_jugadas(self):
        return self.victorias + self.derrotas + self.tablas

    def porcentaje_victorias(self):
        total = self.partidas_jugadas()
        if total:
            return 100 * self.victorias / total
        return 0

    def registrar_resultado(self, puntos):
        if puntos == 1:
            self.victorias += 1
        elif puntos == 0:
            self.derrotas += 1
        else:
            self.tablas += 1

    def resumen(self):
        mensaje = f"{self.nombre} ({self.pais})  ELO {self.elo} | {self.victorias}V {self.derrotas}D {self.tablas}T | {self.porcentaje_victorias():.0f}% victorias"
        return mensaje

# Prueba
a = Jugador("Magnus", "NOR")
b = Jugador("Hikaru", "USA")
print(a.elo, b.elo, Jugador.total_registrados)
# Esperado: 1200 1200 2

1200 1200 2


---

## Paso 4: métodos estáticos

**Concepto: `@staticmethod`.**

1. Crea el **método estático** `variacion_elo(puntos, k)` en `Jugador`. Recibe los `puntos`
   que ha obtenido un jugador en una partida (`1`, `0.5` o `0`) y el factor `k` del ritmo, y
   devuelve **cuánto debe cambiar su ELO**:
   - si `puntos` es `1` (victoria) → devuelve `k` (sube K puntos);
   - si `puntos` es `0` (derrota) → devuelve `-k` (baja K puntos);
   - si `puntos` es `0.5` (tablas) → devuelve `0` (el ELO no cambia).

   Fíjate en que es una **regla pura**: solo depende de sus dos parámetros, no lee ni modifica
   nada del jugador. Por eso no necesita `self` (ni `cls`) y la decoramos con `@staticmethod`.
   Se puede llamar desde la clase, `Jugador.variacion_elo(1, 20)`, sin tener ningún jugador creado.

2. Añade un método de instancia `actualizar_elo(puntos, k)` que use la regla anterior para
   sumar la variación a `self.elo`. Así el jugador es el **dueño de su propio ELO** y nadie
   tiene que hacer cuentas con sus atributos desde fuera.

In [ ]:
class Jugador:
    
    ELO_INICIAL = 1200
    total_registrados = 0
    
    def __init__(self, nombre, pais):
        self.nombre = nombre
        self.pais = pais
        self.elo = Jugador.ELO_INICIAL
        self.victorias = 0
        self.derrotas = 0
        self.tablas = 0
        self.historial = []
        Jugador.total_registrados += 1
        
    @staticmethod
    def variacion_elo(puntos, k):
        if puntos == 1:
            return k
        elif puntos == 0:
            return -k
        else:
            return 0
        
    ##FALTA EL 2.

    def partidas_jugadas(self):
        return self.victorias + self.derrotas + self.tablas

    def porcentaje_victorias(self):
        total = self.partidas_jugadas()
        if total:
            return 100 * self.victorias / total
        return 0

    def registrar_resultado(self, puntos):
        if puntos == 1:
            self.victorias += 1
        elif puntos == 0:
            self.derrotas += 1
        else:
            self.tablas += 1

    def resumen(self):
        mensaje = f"{self.nombre} ({self.pais})  ELO {self.elo} | {self.victorias}V {self.derrotas}D {self.tablas}T | {self.porcentaje_victorias():.0f}% victorias"
        return mensaje

# Prueba
print(Jugador.variacion_elo(1, 20))  # 20
print(Jugador.variacion_elo(0.5, 20))  # 0

j = Jugador("Magnus", "NOR")
j.actualizar_elo(1, 20)  # gana una partida con K=20
print(j.elo)  # 1220

---

## Paso 5: la clase `Ritmo`

**Concepto: una clase pequeña para sustituir cadenas de `if/elif`.**

Las funciones `factor_k(ritmo)` y `tiempo_por_jugador(ritmo)` del programa lineal son dos cadenas de `if/elif`
sobre el mismo string. Si mañana añadimos el ritmo *rapid*, hay que tocar **las dos**.

En lugar de un string, un ritmo va a ser un **objeto** que lleva sus datos consigo. Crea la
clase `Ritmo` con:

- Atributos de instancia: `nombre`, `k` y `tiempo`.
- Un método `describir()` que devuelva `"blitz, 5 min"`.

Después crea **tres objetos**: `BULLET` (K=10, "1 min"), `BLITZ` (K=20, "5 min") y
`CLASICA` (K=40, "90 min"). Los escribimos en mayúsculas porque son constantes del programa.

Finalmente **añade un cuarto ritmo `RAPID`** (K=30, "15 min") y fíjate en cuántas líneas de
código *existente* has tenido que modificar. Compáralo con la versión lineal.

In [ ]:
class Ritmo:
    # TODO
    pass


# TODO: crea BULLET, BLITZ, CLASICA (y luego RAPID)


# Prueba
for ritmo in (BULLET, BLITZ, CLASICA):
    print(f"{ritmo.nombre:<8} K={ritmo.k:<3} -> {ritmo.describir()}")
# Esperado:
# bullet   K=10  -> bullet, 1 min
# blitz    K=20  -> blitz, 5 min
# clasica  K=40  -> clasica, 90 min

---

## Paso 6: la clase `Partida`

**Concepto: una clase para agrupar datos que viajan juntos.**

Ahora mismo una partida es una tupla `(blancas, negras, resultado, ritmo)`: para saber qué
hay en cada posición tienes que recordarlo de memoria. Crea la clase `Partida` con:

- Atributos de instancia: `blancas` y `negras` (objetos `Jugador`), `resultado` (string) y
  `ritmo` (objeto `Ritmo`). Fíjate: ya no guardamos *nombres*, guardamos **los objetos**.
- En `__init__`, valida que el resultado sea `"1-0"`, `"0-1"` o `"1/2-1/2"`; si no, lanza `ValueError`.
- `puntos()` → devuelve la tupla `(puntos_blancas, puntos_negras)`: `(1, 0)`, `(0, 1)` o `(0.5, 0.5)`.
- `rival_de(jugador)` → devuelve el otro jugador de la partida.
- `color_de(jugador)` → devuelve `"blancas"` o `"negras"` según el color con que jugó.
- `describir_para(jugador)` → devuelve la línea del historial desde el punto de vista de ese jugador:
  `"vs Hikaru   con blancas  1-0      [blitz, 5 min]"`.

In [ ]:
class Partida:
    # TODO
    pass


# Prueba
magnus = Jugador("Magnus", "NOR")
hikaru = Jugador("Hikaru", "USA")
p = Partida(magnus, hikaru, "1-0", BLITZ)

print(p.puntos())  # (1, 0)
print(p.rival_de(magnus).nombre, p.color_de(magnus))  # Hikaru blancas
print(p.describir_para(hikaru))  # vs Magnus   con negras   1-0      [blitz, 5 min]

try:
    Partida(magnus, hikaru, "2-0", BLITZ)
except ValueError as e:
    print("ValueError:", e)

---

## Paso 7: la clase `Plataforma`

**Concepto: un objeto que contiene y coordina otros objetos (composición).**

Ya tenemos las piezas: `Jugador`, `Ritmo` y `Partida`. Falta el objeto que las coordina y que
sustituye a las variables globales `jugadores` y `partidas` y a las funciones sueltas.

Crea la clase `Plataforma` con:

- Atributos de instancia: `jugadores` (diccionario `nombre → Jugador`) y `partidas` (lista de `Partida`).
- `registrar(nombre, pais)` → crea un `Jugador` y lo guarda. Si ya existe, imprime el aviso y no hace nada.
- `buscar(nombre)` → devuelve el `Jugador` con ese nombre, o lanza `ValueError` si no está registrado.
- `jugar(nombre_blancas, nombre_negras, resultado, ritmo)` → recibe **nombres** y un objeto `Ritmo`:
  1. Busca los dos jugadores (con `buscar`) y comprueba que son distintos.
  2. Crea la `Partida` (que ya valida el resultado).
  3. Actualiza estadísticas (`registrar_resultado`) y ELO (`actualizar_elo`) de ambos.
  4. Añade la partida a `self.partidas` y al `historial` de cada jugador.
- `ranking()` → devuelve la lista de jugadores ordenada por ELO descendente.
- `mostrar_ranking()` y `mostrar_historial(nombre)` → imprimen lo mismo que la versión lineal.

Fíjate en lo poco que tiene que hacer la plataforma: casi todo es **pedir a cada objeto que
haga su parte** (`jugador.resumen()`, `partida.puntos()`, `partida.describir_para(...)`).

In [ ]:
class Plataforma:
    # TODO
    pass


# --- Programa principal (versión POO) ---
plataforma = Plataforma()
plataforma.registrar("Magnus", "NOR")
plataforma.registrar("Hikaru", "USA")
plataforma.registrar("Ding", "CHN")
plataforma.registrar("Judit", "HUN")
plataforma.registrar("Magnus", "NOR")  # duplicado

plataforma.jugar("Magnus", "Hikaru", "1-0", BLITZ)
plataforma.jugar("Ding", "Judit", "1/2-1/2", CLASICA)
plataforma.jugar("Hikaru", "Ding", "1-0", BULLET)
plataforma.jugar("Judit", "Magnus", "0-1", CLASICA)
plataforma.jugar("Hikaru", "Judit", "0-1", BLITZ)
plataforma.jugar("Magnus", "Ding", "1/2-1/2", BULLET)

plataforma.mostrar_ranking()
plataforma.mostrar_historial("Magnus")
plataforma.mostrar_historial("Judit")

print(f"\nTotal de partidas en la plataforma: {len(plataforma.partidas)}")
# La salida debe coincidir EXACTAMENTE con la del programa lineal

---

## Reflexión final

Compara las dos versiones y responde:

1. Si quisieras añadir un nuevo ritmo de juego, ¿cuántos sitios tocas en cada versión?
2. ¿Dónde vive ahora la regla "un jugador no puede jugar contra sí mismo"? ¿Y la fórmula del ELO?
   ¿Y la validación del resultado?
3. En la versión lineal cualquiera podía hacer `jugadores["Magnus"]["elo"] = 9999`. ¿Puede
   seguir pasando en la versión POO? ¿Qué convención de Python usarías para *desaconsejarlo*?
4. `Partida` guarda **objetos** `Jugador`, no nombres. ¿Qué ventaja te ha dado eso al escribir
   `describir_para`? ¿Qué habría pasado si guardara strings?
5. ¿Qué parte de la versión POO tiene **más** líneas que la lineal? ¿Compensa?